In [12]:
import cv2
import math
import cvzone
import time
import requests
import json
from ultralytics import YOLO

In [13]:
class Crowd_alert_system:
    
    def __init__(self, video_source=0, threshold=15, location="Hospital LOBBY-A", coords="40.7128,-74.0060"):
            # Configuration
            self.video_source = video_source       
            self.threshold = threshold            
            self.location = location
            self.coords = coords
            
        
            print("Loading YOLOv8 model...")
            self.model = YOLO("yolov8n.pt")
            
            self.webhook_url = "http://127.0.0.1:5000/api/alerts"
            
            self.last_alert_time = 0
            self.alert_cooldown = 5
            self.class_names = self.model.names
            
            
    def send_notification(self, count):
            """Builds the JSON payload and sends it to the notification server."""
            payload = {
                "priority": "CRITICAL" if count > (self.threshold * 1.5) else "WARNING",
                "location_name": self.location,
                "coordinates": self.coords,
                "head_count": count,
                "threshold": self.threshold,
                "timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
                "message": f"Crowd threshold exceeded in {self.location}. Current count: {count}."
            }
            
            try:
            
                print(f"\n[ALERT FIRED] -> Sending payload: {json.dumps(payload, indent=2)}")
                response = requests.post(self.webhook_url, json=payload,timeout=10)
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"Failed to send alert: {e}")
                
    def run(self):
            """Main loop to capture video, run inference, and trigger alerts."""
            cap = cv2.VideoCapture(self.video_source)
            
            if not cap.isOpened():
                print("Error: Could not open video source.")
                return
    
            print(f"Starting monitoring for {self.location}... Press 'q' to quit.")        
            
            frame_number = 0
                
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_number += 1
                if frame_number % 2 != 0:
                    continue


                results = self.model(frame,classes=[0],stream=True)

                person_count = 0

                for r in results:
                    boxes = r.boxes

                    for b in boxes:
                        x1, y1, x2, y2 = map(int, b.xyxy[0])

                        conf = float(b.conf[0])
                        cls_id = int(b.cls[0])
                    

                        label = self.class_names[cls_id]

                        if label == "person":
                            person_count += 1

                            w, h = x2 - x1, y2 - y1

                            
                            cvzone.cornerRect(
                                frame,
                                (x1, y1, w, h),
                                l=15,
                                colorC=(255, 0, 255),
                                colorR=(0, 255, 0),
                                t=2
                            )

                            cvzone.putTextRect(
                                frame,
                                f"{label} {conf:.2f} ",
                                (max(0, x1), max(35, y1)),
                                scale=1,
                                thickness=2,
                                colorR=(0, 255, 0)
                            )

                
                if person_count >= self.threshold:
                    status = "CROWDED AREA"
                    color = (0, 0, 255)
                    current_time = time.time() 
                    
                    if (current_time - self.last_alert_time) > self.alert_cooldown:
                        self.send_notification(person_count)
                        self.last_alert_time = current_time 
                else:
                    status = "NORMAL AREA"
                    color = (0, 255, 0)  

                
                cv2.putText(
                    frame,
                    f"People Count: {person_count}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 255),
                    2,
                    
                )

                cv2.putText(
                    frame,
                    status,
                    (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    color,
                    3
                )

                cv2.imshow("Crowd Detection", frame)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

            cap.release()
            cv2.destroyAllWindows()
            
if __name__ == "__main__":

    system = Crowd_alert_system(
        video_source=0, 
        threshold=5, 
        location="Airport", 
        coords="28.5562,77.1000"
    )
    system.run()            

Loading YOLOv8 model...
Starting monitoring for Airport... Press 'q' to quit.

0: 480x640 5 persons, 113.7ms
Speed: 1.9ms preprocess, 113.7ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

[ALERT FIRED] -> Sending payload: {
  "priority": "WARNING",
  "location_name": "Airport",
  "coordinates": "28.5562,77.1000",
  "head_count": 5,
  "threshold": 5,
  "timestamp": "2026-09-02 14:41:02",
  "message": "Crowd threshold exceeded in Airport. Current count: 5."
}

0: 480x640 4 persons, 111.9ms
Speed: 4.8ms preprocess, 111.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 97.4ms
Speed: 1.8ms preprocess, 97.4ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 97.3ms
Speed: 2.2ms preprocess, 97.3ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 108.1ms
Speed: 2.4ms preprocess, 108.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 48

KeyboardInterrupt: 

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

False
CPU
